## import

In [4]:
import msea
# %pip install msea
from msea import SetLibrary
import numpy
import scipy
import pandas as pd
from pprint import pprint
# %pip install biom-format
from biom import load_table  # reqired for parsing BIOM formated dataset
from skbio.stats.composition import ancom
# %pip install scikit-bio


We started with loading and preparing 16S dataset, which can be downloaded from [Qiita](https://qiita.ucsd.edu/study/description/10483#):

In [2]:
# %pip install numpy
%pip install --upgrade scikit-bio

Note: you may need to restart the kernel to use updated packages.


In [9]:
table = load_table("../csv_files/10483_45784_otu_table.biom")

print("biom table shape: ", table.shape)  # OTUs x samples
# Optionally normalize data on the sample axis -> relative abundance
# table.norm(axis='sample', inplace=True)

# Load metadata
meta_df = pd.read_csv(
    '../csv_files/10483_20230206-075026.txt',
    sep='\t',
    index_col='sample_name')
meta_df = meta_df.loc[table.ids()]


def collapse_f(id_, md):
    # collapse OTUs to genus-level
    genus_idx = 5
    return '; '.join(md['taxonomy'][:genus_idx + 1])


table_g = table.collapse(collapse_f, axis='observation')
print(table_g.shape)

# check the microbial species:
print(table_g.ids(axis='observation')[:5])
# convert to a pd.DataFrame
df_g = pd.DataFrame(
    table_g.matrix_data.toarray().T,
    columns=table_g.ids(axis='observation'),
    index=table_g.ids()
)
# Select a subset of samples from genotype 'Thy1-aSyn' for further analysis
meta_df_sub = meta_df.loc[meta_df['genotype'] == 'thy1-asyn']
df_g_sub = df_g.loc[meta_df['genotype'] == 'thy1-asyn']
print(meta_df_sub['donor_status'].value_counts())

meta_df_sub

df_g_sub

biom table shape:  (3397, 453)
(276, 453)
['k__Bacteria; p__Verrucomicrobia; c__Verrucomicrobiae; o__Verrucomicrobiales; f__Verrucomicrobiaceae; g__Akkermansia'
 'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__; g__'
 'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__'
 'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__Oscillospira'
 'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__Ruminococcus']
donor_status
healthy    106
pd          94
Name: count, dtype: int64


,k__Bacteria; p__Verrucomicrobia; c__Verrucomicrobiae; o__Verrucomicrobiales; f__Verrucomicrobiaceae; g__Akkermansia,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__; g__,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__Oscillospira,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__Ruminococcus,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__Dorea,k__Bacteria; p__Firmicutes; c__Bacilli; o__Lactobacillales; f__Streptococcaceae; g__Streptococcus,k__Bacteria; p__Proteobacteria; c__Deltaproteobacteria; o__Myxococcales; f__0319-6G20; g__,k__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Xanthomonadales; f__Xanthomonadaceae; g__Stenotrophomonas,...,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__Robinsoniella,k__Bacteria; p__Actinobacteria; c__Actinobacteria; o__Actinomycetales; f__Micrococcaceae; g__Renibacterium,k__Bacteria; p__Actinobacteria; c__Actinobacteria; o__Actinomycetales; f__Cellulomonadaceae; g__Cellulomonas,k__Bacteria; p__Proteobacteria; c__Betaproteobacteria; o__Burkholderiales; f__Comamonadaceae; g__Leptothrix,k__Bacteria; p__Fusobacteria; c__Fusobacteriia; o__Fusobacteriales; f__Leptotrichiaceae; g__Sneathia,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__EtOH8; g__,k__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Pseudomonadales; f__Moraxellaceae; g__,k__Bacteria; p__Bacteroidetes; c__Flavobacteriia; o__Flavobacteriales; f__[Weeksellaceae]; g__,k__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Legionellales; f__Coxiellaceae; g__Rickettsiella,k__Bacteria; p__Proteobacteria; c__Alphaproteobacteria; o__Rhodobacterales; f__Rhodobacteraceae; g__Rhodobacter
10483.donor1.HC,218.083333,1.512195,0.967433,0.205357,0.048780,5.405622,4.743902,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483.recip.647.ASO.PD6.D7,410.000000,0.324390,0.957854,1.035714,0.365854,2.915663,0.146341,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483.recip.386.WT.PD2.D21,741.750000,2.660976,12.260536,0.401786,0.341463,1.242972,0.073171,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483.recip.390.ASO.PD2.D21,469.000000,0.321951,5.687739,0.241071,0.256098,0.236948,0.036585,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483.recip.456.ASO.HC3.D21,246.666667,0.456098,0.639847,0.473214,0.012195,0.797189,0.390244,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10483.recip.316WT.HC1.D49,1.833333,0.043902,0.126437,0.142857,0.012195,0.046185,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483.recip.325.ASO.PD1.D49,0.000000,0.000000,0.001916,0.008929,0.000000,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483.recip.467.WT.PD3.D14,0.000000,0.000000,0.000000,0.008929,0.000000,0.002008,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10483.recip.560.ASO.HC4.D21,0.000000,0.000000,0.000000,0.000000,0.000000,0.002008,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
df_g_sub + 1

,k__Bacteria; p__Verrucomicrobia; c__Verrucomicrobiae; o__Verrucomicrobiales; f__Verrucomicrobiaceae; g__Akkermansia,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__; g__,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__Oscillospira,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Ruminococcaceae; g__Ruminococcus,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__Dorea,k__Bacteria; p__Firmicutes; c__Bacilli; o__Lactobacillales; f__Streptococcaceae; g__Streptococcus,k__Bacteria; p__Proteobacteria; c__Deltaproteobacteria; o__Myxococcales; f__0319-6G20; g__,k__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Xanthomonadales; f__Xanthomonadaceae; g__Stenotrophomonas,...,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__Robinsoniella,k__Bacteria; p__Actinobacteria; c__Actinobacteria; o__Actinomycetales; f__Micrococcaceae; g__Renibacterium,k__Bacteria; p__Actinobacteria; c__Actinobacteria; o__Actinomycetales; f__Cellulomonadaceae; g__Cellulomonas,k__Bacteria; p__Proteobacteria; c__Betaproteobacteria; o__Burkholderiales; f__Comamonadaceae; g__Leptothrix,k__Bacteria; p__Fusobacteria; c__Fusobacteriia; o__Fusobacteriales; f__Leptotrichiaceae; g__Sneathia,k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__EtOH8; g__,k__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Pseudomonadales; f__Moraxellaceae; g__,k__Bacteria; p__Bacteroidetes; c__Flavobacteriia; o__Flavobacteriales; f__[Weeksellaceae]; g__,k__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Legionellales; f__Coxiellaceae; g__Rickettsiella,k__Bacteria; p__Proteobacteria; c__Alphaproteobacteria; o__Rhodobacterales; f__Rhodobacteraceae; g__Rhodobacter
10483.donor1.HC,219.083333,2.512195,1.967433,1.205357,1.048780,6.405622,5.743902,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10483.recip.647.ASO.PD6.D7,411.000000,1.324390,1.957854,2.035714,1.365854,3.915663,1.146341,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10483.recip.386.WT.PD2.D21,742.750000,3.660976,13.260536,1.401786,1.341463,2.242972,1.073171,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10483.recip.390.ASO.PD2.D21,470.000000,1.321951,6.687739,1.241071,1.256098,1.236948,1.036585,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10483.recip.456.ASO.HC3.D21,247.666667,1.456098,1.639847,1.473214,1.012195,1.797189,1.390244,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10483.recip.316WT.HC1.D49,2.833333,1.043902,1.126437,1.142857,1.012195,1.046185,1.000000,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10483.recip.325.ASO.PD1.D49,1.000000,1.000000,1.001916,1.008929,1.000000,1.000000,1.000000,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10483.recip.467.WT.PD3.D14,1.000000,1.000000,1.000000,1.008929,1.000000,1.002008,1.000000,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
10483.recip.560.ASO.HC4.D21,1.000000,1.000000,1.000000,1.000000,1.000000,1.002008,1.000000,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


Next, we perform DA analysis using ANCOM to identify DA microbes:

In [8]:
ancom_df, percentile_df = ancom(df_g_sub + 1,  # adding pseudocounts
                                meta_df_sub['donor_status'],
                                alpha=0.1,
                                p_adjust='holm-bonferroni')

microbes_DA = ancom_df.loc[ancom_df['Signif']].index
print(microbes_DA)

# retain genus names
microbes_DA = filter(None, [s.split('; ')[-1].lstrip('g__')
                            for s in microbes_DA])
# convert to set
microbes_DA = set(microbes_DA)
print(microbes_DA)



/users/aflemis1/.conda/envs/msea/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:611: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, axis=axis, **kwds)


Index(['k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__',
       'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__Blautia',
       'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Clostridiaceae; g__',
       'k__Bacteria; p__Bacteroidetes; c__Bacteroidia; o__Bacteroidales; f__Rikenellaceae; g__',
       'k__Bacteria; p__Proteobacteria; c__Gammaproteobacteria; o__Enterobacteriales; f__Enterobacteriaceae; g__Proteus',
       'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Eubacteriaceae; g__Pseudoramibacter_Eubacterium',
       'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Lachnospiraceae; g__Clostridium',
       'k__Bacteria; p__Firmicutes; c__Clostridia; o__Clostridiales; f__Veillonellaceae; g__',
       'k__Bacteria; p__Proteobacteria; c__Deltaproteobacteria; o__Desulfovibrionales; f__Desulfovibrionaceae; g__Bilophila'],
      dtype='object')
{'Pseudoramibacter

In [12]:
gmt_filepath = \
    'https://bitbucket.org/wangz10/msea/raw/aee6dd184e9bde152b4d7c2f3c7245efc1b80d23/msea/data/human_genes_associated_microbes/set_library.gmt'

d_gmt = msea.read_gmt(gmt_filepath)

d_gmt

{'A2M': {'Azomonas', 'Borrelia', 'Pseudomonas', 'Salmonella', 'Sodalis'},
 'AAAS': {'Colwellia',
  'Deinococcus',
  'Idiomarina',
  'Neisseria',
  'Pseudidiomarina',
  'Pseudoalteromonas'},
 'AACS': {'Acetobacter',
  'Acinetobacter',
  'Azomonas',
  'Corynebacterium',
  'Enterobacter',
  'Klebsiella',
  'Mycobacterium',
  'Mycoplasma',
  'Pseudomonas',
  'Sodalis',
  'Staphylococcus',
  'Streptomyces',
  'Tetragenococcus'},
 'AARS': {'Agrobacterium',
  'Aquifex',
  'Archaeoglobus',
  'Bacillus',
  'Bdellovibrio',
  'Borrelia',
  'Bradyrhizobium',
  'Burkholderia',
  'Caulobacter',
  'Clostridium',
  'Cyanobacterium',
  'Enterococcus',
  'Geobacillus',
  'Ignicoccus',
  'Metallosphaera',
  'Methanocaldococcus',
  'Methanococcoides',
  'Methanococcus',
  'Methanolobus',
  'Methanopyrus',
  'Methanosaeta',
  'Methanosarcina',
  'Methanosphaera',
  'Methanothermobacter',
  'Methanothermococcus',
  'Microcystis',
  'Mycobacterium',
  'Mycoplasma',
  'Myxococcus',
  'Prochlorococcus',
  'Pse